# Querying by View Name Keyword / Database Name

**Question:** *"How do I query for a view name which has some particular keyword in it?

**Short answer: YES — and it runs today.** Two complementary routes:

| Route | How | Status |
|---|---|---|
| **Client-side** (this notebook) | `backend.find_views(name_contains=..., database=...)` over the crawled `denodo_views` table | Works now, offline, permission-scoped by construction |
| **Server-side** | `POST /search/metadata` with `whereToSearchList=["ELEMENT_NAME"]` (+ `databaseIds` scope) — wrapped as `backend._search_views()` | Wait until dev API auth is restored (M9) |

**I figured out that** the view fields carry **no creator/owner name**. The list-endpoint fields are `{id, name, description, db, elementType, elementSubtype, path, lastModificationDate, fields, deleted}` [STEP0] — the closest built-in provenance is **`lastModificationDate`**. Author-style queries would go through **custom properties** (if LANL defines an Owner/Steward property — probe S5 will check once M9 is fixed), searchable server-side via `PROPERTY_VALUE`.

**M9**: since the migration, the dev Data Catalog REST API rejects valid OAuth bearer tokens with 401 while the browser UI logs in normally — Maxen needs to restore the API's OAuth resource-server configuration. The evidence chain: a demonstrably valid, freshly refreshed token; identical failures through both call paths; the unauthenticated spec endpoint still working; the decisive WWW-Authenticate: Basic realm="Denodo" header showing the server isn't even attempting Bearer validation; and UI session cookies also being rejected. 

In [1]:
# Working folder (must contain denodo.py and test_denodo_step2.py)
import os
WORK_DIR = r"C:/Users/414515/Downloads/Denodo Backend Project/dsi/dsi/backends/Step2_Internal_HTTP_Layer"
%cd {WORK_DIR}

C:\Users\414515\Downloads\Denodo Backend Project\dsi\dsi\backends\Step2_Internal_HTTP_Layer


## 1. Setup — offline backend over the golden fixtures
Same data shapes as the real catalog (incl. the dev `db` field drift and an error-500 view), zero network.

In [2]:
import os, sys
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
sys.modules.pop("denodo", None)
from unittest.mock import patch
from denodo import Denodo
from test_denodo_step2 import fake_request

backend = Denodo(base_url="https://datacatalog-d.lanl.gov", server_id=1,
                 token_provider=lambda: "offline", source_env="dev")
with patch.object(Denodo, "_request", fake_request):
    backend.process_artifacts()
backend.list()

view-details failed for (vlanlchangerequesttask, dataportal): Denodo API error 500 on /denodo-data-catalog/public/api/view-details (synthetic error_500 cohort)


denodo_databases: (1 rows, 4 cols)
denodo_views: (2 rows, 8 cols)
denodo_columns: (1 rows, 5 cols)
denodo_properties: (2 rows, 4 cols)


## 2. "Query for a view name which has some particular keyword in it" 

In [3]:
# Keyword in the VIEW NAME (case-insensitive substring):
backend.find_views(name_contains="inventory")

,view_name,db_name,description,documentation_url,source_system,access_instructions,fetch_status,source_env
0,inventory_daily_balance_fact,dataportal,Balances. See docs .,https://etrm.live/doc123,EBS_LINK,Request,ok,dev


In [4]:
# Case-insensitivity check -- "TASK" matches vlanlchangerequesttask:
backend.find_views(name_contains="TASK")

,view_name,db_name,description,documentation_url,source_system,access_instructions,fetch_status,source_env
0,vlanlchangerequesttask,dataportal,None,None,None,None,error,dev


## 3. "…or a database name (which of course will give you everything)" 

In [5]:
# Everything in one database:
backend.find_views(database="dataportal")

,view_name,db_name,description,documentation_url,source_system,access_instructions,fetch_status,source_env
0,inventory_daily_balance_fact,dataportal,Balances. See docs .,https://etrm.live/doc123,EBS_LINK,Request,ok,dev
1,vlanlchangerequesttask,dataportal,None,None,None,None,error,dev


In [6]:
# Combined: keyword AND database scope
backend.find_views(name_contains="balance", database="dataportal")

,view_name,db_name,description,documentation_url,source_system,access_instructions,fetch_status,source_env
0,inventory_daily_balance_fact,dataportal,Balances. See docs .,https://etrm.live/doc123,EBS_LINK,Request,ok,dev


## 4. Beyond the name: description keywords and cell-level search
`find_views` also matches inside descriptions; `find()` sweeps names, columns, and property values at once.

In [7]:
print("-- description keyword --")
display(backend.find_views(description_contains="balances"))

print("-- catalog-wide sweep: find('balance') --")
for v in backend.find("balance")[:6]:
    print(f"  {v.type:6s} {v.t_name} :: {v.c_name} = {str(v.value)[:50]}")

-- description keyword --


,view_name,db_name,description,documentation_url,source_system,access_instructions,fetch_status,source_env
0,inventory_daily_balance_fact,dataportal,Balances. See docs .,https://etrm.live/doc123,EBS_LINK,Request,ok,dev


-- catalog-wide sweep: find('balance') --
  cell   denodo_views :: ['view_name'] = inventory_daily_balance_fact
  cell   denodo_views :: ['description'] = Balances. See docs .
  cell   denodo_columns :: ['view_name'] = inventory_daily_balance_fact
  cell   denodo_columns :: ['column_name'] = balance_amt
  cell   denodo_columns :: ['description'] = Daily balance
  cell   denodo_properties :: ['view_name'] = inventory_daily_balance_fact


## 5. The server-side twin (parked until M9)
Identical semantics pushed to Denodo — useful for targeted questions without a full crawl. One flag flip + Run All once Maxen restores dev API OAuth.

In [8]:
AUTH_RESTORED = False   # flip to True after M9 is resolved

if AUTH_RESTORED:
    from oauth_manager import OAuthManager
    mgr = OAuthManager.from_env()
    live = Denodo(base_url="https://datacatalog-d.lanl.gov", server_id=1,
                  source_env="dev", token_provider=mgr.get_access_token)

    # keyword in view NAME only:
    print(str(live._search_views("inventory", where=("ELEMENT_NAME",)))[:800])

    # scoped to one database (integer id from the databases endpoint):
    ids = {(d.get("databaseName") or d.get("name")): d.get("id")
           for d in live._list_databases()}
    print("databases:", ids)   # expect THREE per the Marketplace UI
    print(str(live._search_views("inventory", where=("ELEMENT_NAME",),
              database_ids=[i for i in [ids.get("dataportal")] if i]))[:500])
else:
    print("Server-side demo parked until dev API auth is restored (M9).")

Server-side demo parked until dev API auth is restored (M9).


## 6. Summary

- **View-name keyword query: possible and working** — `find_views(name_contains=...)` today; `search/metadata` with `ELEMENT_NAME` server-side once M9 is fixed. Case-insensitive, combinable with database scope.
- **Database-name query: possible and working** — `find_views(database=...)`, or `databaseIds` server-side.
- **Creator/owner: correctly absent from the fields** — the only built-in provenance is `lastModificationDate`. The author pathway is custom properties (searchable via `PROPERTY_VALUE`); whether LANL defines an Owner/Steward property is probe S5's question after M9.
- All client-side queries are inherently **permission-scoped**: the crawl only ever contained what the authenticated user was allowed to see.